# Scope-Aware Tools: LangChain + ZeroID

This notebook shows one simple idea: the same LangChain agent should not always see the same tools.

With ZeroID, the token decides which tools are visible and which actions are still allowed at execution time. We will run the same prompt three times:

1. With a read-only token
2. With a refund-enabled token
3. With the refund token after revocation

The example uses a deterministic local model, so no external LLM API key is required.


In [1]:
import os
import uuid
from dataclasses import dataclass
from typing import Any, Sequence

from highflame.zeroid import ZeroIDClient
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from pydantic import Field

ZEROID_BASE_URL = os.getenv("ZEROID_BASE_URL", "http://localhost:8899")
ZEROID_ADMIN_API_KEY = os.getenv("ZEROID_ADMIN_API_KEY")

client = ZeroIDClient(base_url=ZEROID_BASE_URL, api_key=ZEROID_ADMIN_API_KEY)
client.health()


HealthResponse(status='healthy', service='zeroid', timestamp=datetime.datetime(2026, 4, 15, 1, 57, 46, 697128, tzinfo=TzInfo(UTC)), uptime_ms=924324)

## 1. Bootstrap a demo agent and issue two tokens

We register one support agent, allow it to ever receive three scopes, and then mint two short-lived access tokens from the same API key:

- `docs:read customer:read`
- `docs:read customer:read refund:write`

The important part is that the **agent identity stays the same** while the **capabilities change with the token**.


In [2]:
def bootstrap_demo_agent():
    external_id = f"langchain-support-demo-{uuid.uuid4().hex[:8]}"
    registered = client.agents.register(
        name="LangChain Support Agent",
        external_id=external_id,
        sub_type="tool_agent",
        trust_level="first_party",
        created_by="demo@highflame.ai",
    )
    client.identities.update(
        registered.identity.id,
        allowed_scopes=["docs:read", "customer:read", "refund:write"],
        description="Demo agent for scope-aware tool filtering",
    )

    read_only_token = client.tokens.issue_api_key(
        registered.api_key,
        scope="docs:read customer:read",
    )
    refund_token = client.tokens.issue_api_key(
        registered.api_key,
        scope="docs:read customer:read refund:write",
    )

    print("Agent URI:", registered.identity.wimse_uri)
    print("Read-only scopes:", client.tokens.session(read_only_token.access_token).scopes)
    print("Refund-enabled scopes:", client.tokens.session(refund_token.access_token).scopes)

    return registered, read_only_token.access_token, refund_token.access_token


registered_agent, read_only_token, refund_token = bootstrap_demo_agent()


Agent URI: spiffe://highflame.ai/default/default/agent/langchain-support-demo-a0d743bd
Read-only scopes: ('docs:read', 'customer:read')
Refund-enabled scopes: ('docs:read', 'customer:read', 'refund:write')


## 2. Define a ZeroID runtime context and three tools

Each tool asks ZeroID for a fresh session from the current bearer token and then calls `require_scope(...)` at the tool boundary.

That means tool filtering is helpful for safety and UX, but it is **not** the only control. The tool itself still verifies authorization.


In [3]:
SCOPES_BY_TOOL = {
    "search_docs": "docs:read",
    "read_customer": "customer:read",
    "refund_customer": "refund:write",
}


@dataclass
class ZeroIDLangChainContext:
    label: str
    auth_headers: dict[str, str]


def build_context(label: str, token: str) -> ZeroIDLangChainContext:
    return ZeroIDLangChainContext(
        label=label,
        auth_headers={"Authorization": f"Bearer {token}"},
    )


def current_session(runtime: ToolRuntime[ZeroIDLangChainContext]):
    return client.tokens.session_from_request(runtime.context.auth_headers)


@tool
def search_docs(query: str, runtime: ToolRuntime[ZeroIDLangChainContext]) -> str:
    """Search the support knowledge base for policy answers."""
    session = current_session(runtime)
    session.require_scope("docs:read")
    return (
        f"docs lookup by {runtime.context.label}: "
        "refunds are allowed within 30 days for duplicate charges."
    )


@tool
def read_customer(customer_id: str, runtime: ToolRuntime[ZeroIDLangChainContext]) -> str:
    """Read a customer's account summary before taking action."""
    session = current_session(runtime)
    session.require_scope("customer:read")
    return (
        f"customer {customer_id}: order=ord_1001, "
        "last_charge=149.00 USD, status=eligible"
    )


@tool
def refund_customer(
    customer_id: str,
    amount: str,
    runtime: ToolRuntime[ZeroIDLangChainContext],
) -> str:
    """Submit a refund for an eligible customer order."""
    session = current_session(runtime)
    session.require_scope("refund:write")
    return (
        f"refund submitted by {session.sub} for {customer_id}: "
        f"amount={amount}"
    )


## 3. Create a tiny deterministic model and a scope-filtering middleware

The model below is intentionally tiny and rule-based so the notebook stays easy to run. In a real application you would replace it with your normal LangChain chat model.

The ZeroID integration points stay the same:

- runtime context carries the bearer token
- middleware filters the visible tools
- each tool re-checks the token before doing work


In [4]:
class RuleBasedToolModel(BaseChatModel):
    bound_tools: list[Any] = Field(default_factory=list)

    @property
    def _llm_type(self) -> str:
        return "rule-based-tool-model"

    def bind_tools(
        self,
        tools: Sequence[Any],
        *,
        tool_choice: str | None = None,
        **kwargs: Any,
    ):
        clone = self.model_copy(deep=True)
        clone.bound_tools = list(tools)
        return clone

    def _generate(self, messages: list[BaseMessage], stop=None, run_manager=None, **kwargs: Any) -> ChatResult:
        last = messages[-1]
        if isinstance(last, ToolMessage):
            return ChatResult(
                generations=[
                    ChatGeneration(
                        message=AIMessage(content=last.content)
                    )
                ]
            )

        user_text = ""
        for message in reversed(messages):
            if isinstance(message, HumanMessage):
                user_text = str(message.content)
                break

        available = {tool.name for tool in self.bound_tools}
        lowered = user_text.lower()

        if "refund" in lowered:
            if "refund_customer" in available:
                tool_call = {
                    "name": "refund_customer",
                    "args": {"customer_id": "cust_123", "amount": "149.00"},
                    "id": "call_refund",
                    "type": "tool_call",
                }
                return ChatResult(
                    generations=[
                        ChatGeneration(message=AIMessage(content="", tool_calls=[tool_call]))
                    ]
                )

            if "read_customer" in available:
                tool_call = {
                    "name": "read_customer",
                    "args": {"customer_id": "cust_123"},
                    "id": "call_read_customer",
                    "type": "tool_call",
                }
                return ChatResult(
                    generations=[
                        ChatGeneration(message=AIMessage(content="", tool_calls=[tool_call]))
                    ]
                )

            return ChatResult(
                generations=[
                    ChatGeneration(
                        message=AIMessage(
                            content="I am not authorized to take action for this request."
                        )
                    )
                ]
            )

        tool_call = {
            "name": "search_docs",
            "args": {"query": user_text or "refund policy"},
            "id": "call_search_docs",
            "type": "tool_call",
        }
        return ChatResult(
            generations=[
                ChatGeneration(message=AIMessage(content="", tool_calls=[tool_call]))
            ]
        )


@wrap_model_call
def zeroid_tool_filter(request, handler):
    session = client.tokens.session_from_request(request.runtime.context.auth_headers)
    visible_tools = []
    for tool_obj in request.tools:
        required_scope = SCOPES_BY_TOOL.get(tool_obj.name)
        if required_scope and session.active and session.has_scope(required_scope):
            visible_tools.append(tool_obj)
    return handler(request.override(tools=visible_tools))


support_agent = create_agent(
    model=RuleBasedToolModel(),
    tools=[search_docs, read_customer, refund_customer],
    middleware=[zeroid_tool_filter],
    context_schema=ZeroIDLangChainContext,
)


## 4. Helper to inspect scopes and run the agent

Before each run, we introspect the token and show which tools should be visible. Then we invoke the same agent with the same prompt.


In [5]:
def visible_tools_for(token: str):
    session = client.tokens.session(token)
    if not session.active:
        return []
    return [
        tool_name
        for tool_name, required_scope in SCOPES_BY_TOOL.items()
        if session.has_scope(required_scope)
    ]


def run_demo(label: str, token: str, prompt: str):
    session = client.tokens.session(token)
    print(f"{label}: active={session.active}")
    print(f"{label}: scopes={session.scopes}")
    print(f"{label}: visible_tools={visible_tools_for(token)}")

    result = support_agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        context=build_context(label, token),
    )

    for message in result["messages"]:
        if isinstance(message, AIMessage) and message.tool_calls:
            print("Chosen tool:", message.tool_calls[0]["name"])
        elif isinstance(message, ToolMessage):
            print("Tool result:", message.content)
        elif isinstance(message, AIMessage):
            print("Final answer:", message.content)


prompt = "Customer cust_123 says they were charged twice. Please refund them."


## 5. Same prompt, read-only token

The agent can still inspect customer state, but it cannot see the refund tool.


In [6]:
run_demo("read-only", read_only_token, prompt)


read-only: active=True
read-only: scopes=('docs:read', 'customer:read')
read-only: visible_tools=['search_docs', 'read_customer']
Chosen tool: read_customer
Tool result: customer cust_123: order=ord_1001, last_charge=149.00 USD, status=eligible
Final answer: customer cust_123: order=ord_1001, last_charge=149.00 USD, status=eligible


## 6. Same prompt, refund-enabled token

Nothing about the agent or prompt changed. Only the token changed, so the refund tool becomes visible and executable.


In [7]:
run_demo("refund-enabled", refund_token, prompt)


refund-enabled: active=True
refund-enabled: scopes=('docs:read', 'customer:read', 'refund:write')
refund-enabled: visible_tools=['search_docs', 'read_customer', 'refund_customer']
Chosen tool: refund_customer
Tool result: refund submitted by spiffe://highflame.ai/default/default/agent/langchain-support-demo-a0d743bd for cust_123: amount=149.00
Final answer: refund submitted by spiffe://highflame.ai/default/default/agent/langchain-support-demo-a0d743bd for cust_123: amount=149.00


## 7. Revoke the privileged token and run again

This time the token is inactive, so ZeroID removes every tool from the model's view and the request is denied on the next invocation.


In [8]:
client.tokens.revoke(refund_token)
run_demo("revoked", refund_token, prompt)


revoked: active=False
revoked: scopes=()
revoked: visible_tools=[]
Final answer: I am not authorized to take action for this request.


## What this integration demonstrates

- ZeroID scopes can decide which LangChain tools are even visible to the model.
- Sensitive tools still enforce scopes at execution time, so filtering is not the only control.
- The same agent identity can operate in different modes just by changing tokens.
- Revocation takes effect on the next call, which matters for long-running agent systems.

To plug this into a real application, replace `RuleBasedToolModel` with your actual LangChain model. The ZeroID integration stays the same: pass the token in runtime context, filter tools from scopes, and call `session.require_scope(...)` inside each tool.
